In [1]:
import os
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, RocCurveDisplay, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import plotly.express as px
import matplotlib.pyplot as plt
import bioframe as bf

pd.options.display.max_columns = 100

# Functions

In [2]:
def training(df, features, test_chrom=None, test_sample=None):
    """ Trains a model returns classifications. """

    if test_chrom != None:
        X_train = df.loc[~df['chrom'].isin(test_chrom), features]
        X_test = df.loc[df['chrom'].isin(test_chrom), features]
        y_train = df.loc[~df['chrom'].isin(test_chrom), 'confirmed']
        y_test = df.loc[df['chrom'].isin(test_chrom), 'confirmed']
    elif test_sample != None:
        X_train = df.loc[~df['sample'].isin(test_sample), features]
        X_test = df.loc[df['sample'].isin(test_sample), features]
        y_train = df.loc[~df['sample'].isin(test_sample), 'confirmed']
        y_test = df.loc[df['sample'].isin(test_sample), 'confirmed']
    else:
        return None

    RF = RandomForestClassifier(n_estimators=100)
    RF.fit(X_train, y_train)

    predictions = RF.predict(X_test)
    proba = RF.predict_proba(X_test)
    X_test['confirmed'] = y_test
    X_test['pred_dicast'] = predictions
    X_test['qual_dicast'] = proba[:, 1]
    result = df[['id', 'sample', 'method', 'type', 'chrom', 'start', 'end', 'filter', 'qual']].merge(X_test[['confirmed', 'pred_dicast', 'qual_dicast']], left_index=True, right_index=True)

    return result

In [3]:
def load_sample_data(SAMPLE, REF):
    """ Load sample data from a single sample. """
    
    FEATURE_DIR = f'/confidential/FamilyR13/DATA/10x/sv_compare/results/{SAMPLE}_{REF}/ensemble'
    df_raw = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.raw.tsv', sep='\t')
    df_ref = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.ref.tsv', sep='\t', low_memory=False)

    filenames_aln_ill = glob(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.aln.ill.*.tsv')
    df_aln_ill = pd.concat([pd.read_csv(f, sep='\t') for f in filenames_aln_ill], ignore_index=True)

    df = df_raw.merge(df_ref.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='left')
    df = df.merge(df_aln_ill.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='inner')

    return df

In [4]:
# Mark variants which have been found by multiple callers

def extract_overlap_ids(df1, df2):
    """ Extracts SV IDs of overlapping variants"""

    closest_intervals = bf.closest(df1, df2, suffixes=('_1','_2'))
    closest_intervals = closest_intervals.dropna(subset=['id_1', 'id_2']).reset_index(drop=True)
    closest_intervals['diff_start'] = abs(closest_intervals['start_1'] - closest_intervals['start_2'])
    closest_intervals['diff_end'] = abs(closest_intervals['end_1'] - closest_intervals['end_2'])
    closest_intervals['diff_size'] = closest_intervals.apply(lambda x: min([x['size_1'], x['size_2']]) / max([x['size_1'], x['size_2']]), axis=1)
    overlapping_svs = closest_intervals[(closest_intervals['diff_start'] < 50) & (closest_intervals['diff_end'] < 50) & (closest_intervals['diff_size'] > 0.7)].copy()
    
    return overlapping_svs[['id_1', 'id_2']].reset_index(drop=True)

In [5]:
def merge_overlapping_svs(result, method_dfs, sample, methods):
    result_sample = result[result['sample'] == sample].copy().reset_index(drop=True)
    for i in range(len(methods)):
        for j in range(i+1, len(methods)):
            method_df_sample_1 = method_dfs[methods[i]][method_dfs[methods[i]]['sample'] == sample].copy().reset_index(drop=True)
            method_df_sample_2 = method_dfs[methods[j]][method_dfs[methods[j]]['sample'] == sample].copy().reset_index(drop=True)
            overlap_ids = extract_overlap_ids(method_df_sample_1, method_df_sample_2)
            for k in range(len(overlap_ids)):
                result_sample.loc[result_sample['id'] == overlap_ids['id_1'][k], 'qual_' + methods[j]] = result_sample.loc[result_sample['id'] == overlap_ids['id_2'][k], 'qual_' + methods[j]].values
                result_sample.loc[result_sample['id'] == overlap_ids['id_2'][k], 'qual_' + methods[i]] = result_sample.loc[result_sample['id'] == overlap_ids['id_1'][k], 'qual_' + methods[i]].values
            
    # Normalize Qualities
    for method in methods:
        result_sample['qual_' + method] = result_sample['qual_' + method] / result_sample['qual_' + method].max()

    return result_sample

In [6]:
def check_caller_support(row, min_num_callers):
    caller_count = 0
    for qual in row:
        if qual != 0:
            caller_count += 1
    if caller_count >= min_num_callers:
        return row.sum()
    else:
        return 0

# Script

In [7]:
# PARAMETERS
DATE = '20230123'
SAMPLES = ['17-08', '176-98', '146-97']
REF = 'hg38'
TYPE = 'INS'
CHROMS = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 
          'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX']

In [8]:
# Load Data
dfs = []
for SAMPLE in SAMPLES:
    df = load_sample_data(SAMPLE, REF)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [9]:
# Filter NaNs
features = ['size'] + list(df.columns[13:])
df = df.dropna(subset=features).copy().reset_index(drop=True)

# Select SV Type
df = df[df['type'] == TYPE].copy().reset_index(drop=True)

## Create set for manual curation

In [10]:
methods = list(df['method'].unique())
for chrom in tqdm(CHROMS):
    result = training(df, features, test_chrom=[chrom])
    curr_fp_svs = list(result.loc[(result['confirmed'] == 0) & (result['pred_dicast'] == 1), 'id'])
    curr_fn_svs = list(result.loc[(result['confirmed'] == 1) & (result['pred_dicast'] == 0), 'id'])

    method_dfs = dict()
    methods = list(result['method'].unique())
    for method in methods:
        result['qual_' + method] = 0
        result.loc[result['method'] == method, 'qual_' + method] = result.loc[result['method'] == method, 'qual']
        method_dfs[method] = df.loc[df['method'] == method, ['id', 'sample', 'method', 'chrom', 'start', 'end', 'type', 'size']].copy()
    result.drop('qual', axis=1, inplace=True)

    result = pd.concat([merge_overlapping_svs(result, method_dfs, sample, methods) for sample in SAMPLES], ignore_index=True)
    result['qual_one caller support'] = result[['qual_' + method for method in methods]].sum(axis=1)
    result['qual_one caller support'] = result['qual_one caller support'].apply(lambda x: np.round(x, 2))
    for method in methods:
        result['qual_' + method] = result['qual_' + method].apply(lambda x: np.round(x, 2))

    # ID needs to be fourth column for Jakobs webapp
    result = result[list(result.columns[1:4]) + ['id'] + list(result.columns[4:])]

    result = result[(result['confirmed'] == 0) & ((result['qual_dicast'] > 0.4) | (result['qual_one caller support'] > 0.4))]
    for sample in SAMPLES:
        if not os.path.isdir(f'/confidential/tGenVar/sv_manual_curation/{sample}/{TYPE}/{chrom}/images'):
            os.makedirs(f'/confidential/tGenVar/sv_manual_curation/{sample}/{TYPE}/{chrom}/images')
        result_sample = result[result['sample'] == sample].copy().reset_index(drop=True)
        result_sample.to_csv(f'/confidential/tGenVar/sv_manual_curation/{sample}/{TYPE}/{chrom}/{DATE}_{sample}_{TYPE}_{chrom}.tsv', index=False, sep='\t')

  0%|          | 0/23 [00:08<?, ?it/s]


KeyError: "['qual_lumpy'] not in index"

In [18]:
list(result.columns[1:4]) + ['id'] + list(result.columns[4:])

['sample',
 'method',
 'type',
 'id',
 'chrom',
 'start',
 'end',
 'filter',
 'confirmed',
 'pred_dicast',
 'qual_dicast',
 'qual_delly',
 'qual_manta',
 'qual_one caller support']

# Evaluate Classifier

In [152]:
chrom_train = ['chr1', 'chr3', 'chr4', 'chr5', 'chr7', 'chr8', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr19', 'chr20', 'chr21']
chrom_test = ['chr2', 'chr6', 'chr9', 'chr18', 'chr22']
sample_test = ['176-98']

In [153]:
result = training(df, features, test_sample=sample_test)

In [154]:
method_dfs = dict()
methods = list(result['method'].unique())
for method in methods:
    result['qual_' + method] = 0
    result.loc[result['method'] == method, 'qual_' + method] = result.loc[result['method'] == method, 'qual']
    method_dfs[method] = df.loc[df['method'] == method, ['id', 'sample', 'method', 'chrom', 'start', 'end', 'type', 'size']].copy()
result.drop('qual', axis=1, inplace=True)

In [155]:
result = pd.concat([merge_overlapping_svs(result, method_dfs, sample, methods) for sample in SAMPLES], ignore_index=True)

In [156]:
result['qual_one caller support'] = result[['qual_' + method for method in methods]].sum(axis=1)
result['qual_two caller support'] = result.apply(lambda x: check_caller_support(x[['qual_' + method for method in methods]], 2), axis=1)
result['qual_three caller support'] = result.apply(lambda x: check_caller_support(x[['qual_' + method for method in methods]], 3), axis=1)

In [157]:
methods_extended = ['dicast'] + methods + ['one caller support', 'two caller support', 'three caller support']
pr_rc_dict = {'method' : [], 'precision' : [], 'recall' : []}
for method in methods_extended:
    precision, recall, _ = precision_recall_curve(result['confirmed'], result['qual_' + method])
    pr_rc_dict['method'].extend([method] * (len(precision) - 1))
    pr_rc_dict['precision'].extend(precision[1:])
    pr_rc_dict['recall'].extend(recall[1:])
pr_rc_df = pd.DataFrame(pr_rc_dict)

In [158]:
colors = ['black', '#1f77b4', '#ff7f0e', 'darkred', '#1f77b4', '#ff7f0e', 'darkred']
dash = ['solid', 'solid', 'solid', 'dot', 'dot', 'dot', 'dot']
circle_bg_white = [0, 0, 0, 1, 1, 1, 1]
fig = px.line(x='recall', y='precision', color='method', 
              data_frame=pr_rc_df, 
              title='Precision-Recall Curve without Manual Curation (' + TYPE + ')', line_dash='method', 
              line_dash_sequence=dash,
              color_discrete_sequence=colors)

for i, method in enumerate(methods_extended):
    x = pr_rc_df[pr_rc_df['method'] == method].reset_index(drop=True).loc[0, 'recall']
    y = pr_rc_df[pr_rc_df['method'] == method].reset_index(drop=True).loc[0, 'precision']
    if circle_bg_white[i] == 1:
        fig.add_shape(type='circle', xref='x', yref='y', x0=x-0.005, y0=y-0.015, x1=x+0.005, y1=y+0.005, line_color=colors[i], line_width=2, opacity=1, fillcolor='white')
    else:
        fig.add_shape(type='circle', xref='x', yref='y', x0=x-0.005, y0=y-0.015, x1=x+0.005, y1=y+0.005, line_color=colors[i], line_width=2, opacity=1, fillcolor=colors[i])

fig.update_layout(plot_bgcolor='white', xaxis_title='Recall', yaxis_title='Precision', xaxis_linecolor='black', yaxis_linecolor='black')
fig.update_traces(line=dict(width=2))
fig.update_xaxes(ticks='outside', tickcolor='black', tickwidth=1, ticklen=5, gridcolor='lightgray', gridwidth=0.5, range=[0, 1.1])
fig.update_yaxes(ticks='outside', tickcolor='black', tickwidth=1, ticklen=5, gridcolor='lightgray', gridwidth=0.5, range=[0, 1.1])
fig.write_image('figures/pr_curve_no_curation_' + TYPE + '.png', width=1200, height=650, scale=3)